# ML03：autograd、训练诊断与 LoRA 梯度

从手推 CE 梯度到 optimizer、train/eval、freeze 和 LoRA 初始化。

**运行方式**：上传本 `.ipynb` 到 Google Colab，连接 Python runtime，从上到下运行。
默认 CPU 即可；本 notebook 自带代码和合成数据，不需要克隆仓库、上传论文、挂载 Drive 或配置本地环境。
所有输出留空，首次运行后才产生实际结果；不要把材料准备状态当作已复现实验。

如果所选云端 runtime 缺少包，可在 Colab 新单元执行 `%pip install numpy matplotlib pandas scikit-learn torch`，
然后重启 runtime。不要为了本课程在本机安装依赖。


## 1. 环境与随机性
记录版本和 seed。换 seed 时从本单元重新运行；图中用英文标签避免依赖额外中文字体。


In [ ]:
import json
import math
import platform
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
print('Python:', platform.python_version(), '| NumPy:', np.__version__, '| seed:', SEED)
import torch
from torch import nn
import torch.nn.functional as F
torch.manual_seed(SEED)
torch.set_num_threads(2)
print('torch:', torch.__version__, '| teaching device: CPU')


## 2. CE 梯度：手算、autograd、有限差分
logits=[B,C]、整数 targets=[B]。Loss 对 batch mean，因此梯度除以 B。


In [ ]:
logits = torch.tensor([[.3, -.2, .8], [.1, .7, -.4]], dtype=torch.float64, requires_grad=True)
targets = torch.tensor([2, 1])
objective = F.cross_entropy(logits, targets)
objective.backward()
hand = (logits.detach().softmax(-1) - F.one_hot(targets, 3)) / len(targets)
eps = 1e-5; numerical = torch.zeros_like(logits)
for i in range(logits.shape[0]):
    for j in range(logits.shape[1]):
        delta = torch.zeros_like(logits); delta[i, j] = eps
        numerical[i, j] = (F.cross_entropy(logits.detach()+delta, targets) -
                            F.cross_entropy(logits.detach()-delta, targets))/(2*eps)
assert torch.allclose(logits.grad, hand, atol=1e-10)
assert torch.allclose(logits.grad, numerical, atol=1e-7)
results = {'seed': SEED, 'ce_hand_gradient_error': float((logits.grad-hand).abs().max()),
           'ce_finite_difference_error': float((logits.grad-numerical).abs().max())}
print(results)
bad_prediction, bad_target = torch.zeros(4, 1), torch.ones(4)
print('Broadcast bug:', (bad_prediction-bad_target).shape, 'expected [4,1]')


## 3. 同一参数初始化，SGD 与 AdamW 的训练轨迹
只在 train 更新；validation loss 用于诊断。训练预算、weight decay 与学习率不同，结果不能当 optimizer 普遍排名。


In [ ]:
import copy
data_generator = torch.Generator().manual_seed(SEED)
x = torch.randn(500, 4, generator=data_generator)
y = (x[:, :1] + .6*x[:, 1:2]**2 + .1*torch.randn(500, 1, generator=data_generator))
xtr, xva, ytr, yva = x[:350], x[350:], y[:350], y[350:]
torch.manual_seed(SEED)
initial = nn.Sequential(nn.Linear(4, 32), nn.Tanh(), nn.Linear(32, 1))
trajectories = {}; final_metrics = {}
for name in ('SGD', 'AdamW'):
    model = copy.deepcopy(initial)
    optimizer = (torch.optim.SGD(model.parameters(), lr=.04, momentum=.9) if name=='SGD'
                 else torch.optim.AdamW(model.parameters(), lr=.01, weight_decay=.01))
    trace = []
    for step in range(300):
        model.train(); optimizer.zero_grad(set_to_none=True)
        loss_value = F.mse_loss(model(xtr), ytr)
        loss_value.backward(); optimizer.step()
        if step % 5 == 0:
            model.eval()
            with torch.no_grad():
                trace.append([step, float(F.mse_loss(model(xtr), ytr)), float(F.mse_loss(model(xva), yva))])
    trajectories[name] = np.asarray(trace)
    final_metrics[name] = {'train_mse': trace[-1][1], 'validation_mse': trace[-1][2]}
for name, trace in trajectories.items():
    plt.plot(trace[:, 0], trace[:, 2], label=name)
plt.xlabel('step'); plt.ylabel('validation MSE'); plt.legend(); plt.show()
results['optimization'] = final_metrics


## 4. train/eval 不等于有无 autograd
Dropout 的 eval 输出确定；BatchNorm 的 running stats 在常见默认配置下仅在 train 更新。


In [ ]:
dropout = nn.Dropout(.5); inputs = torch.ones(32, 8)
dropout.train(); train_a, train_b = dropout(inputs), dropout(inputs)
dropout.eval(); eval_a, eval_b = dropout(inputs), dropout(inputs)
assert torch.equal(eval_a, eval_b)
batch_norm = nn.BatchNorm1d(8)
batch_norm.train(); batch_norm(inputs * 3)
running_after_train = batch_norm.running_mean.clone()
batch_norm.eval(); batch_norm(inputs * 20)
assert torch.equal(batch_norm.running_mean, running_after_train)
leaf = torch.tensor(2., requires_grad=True)
(leaf**2).backward(); first_gradient = float(leaf.grad)
(leaf**2).backward(); accumulated = float(leaf.grad)
assert accumulated == 2*first_gradient
print('Dropout train outputs differ:', not torch.equal(train_a, train_b))
print('Backward accumulates:', first_gradient, '->', accumulated)
results['gradient_accumulation'] = [first_gradient, accumulated]


## 5. LoRA：B=0 时，首步 A 的零梯度不等于 frozen
base weight 冻结，A 随机、B 全零。只检查一步的梯度与参数量，不训练或下载大模型。


In [ ]:
class TinyLoRA(nn.Module):
    def __init__(self, in_dim=6, out_dim=4, rank=2, alpha=2):
        super().__init__()
        self.base = nn.Linear(in_dim, out_dim, bias=False)
        self.base.requires_grad_(False)
        self.A = nn.Parameter(.1*torch.randn(rank, in_dim))
        self.B = nn.Parameter(torch.zeros(out_dim, rank))
        self.scale = alpha/rank
    def forward(self, value):
        return self.base(value) + self.scale*(value @ self.A.T @ self.B.T)
lora = TinyLoRA()
sample = torch.randn(12, 6); target = torch.randn(12, 4)
F.mse_loss(lora(sample), target).backward()
a_grad = float(lora.A.grad.norm()); b_grad = float(lora.B.grad.norm())
assert lora.base.weight.grad is None and a_grad == 0.0 and b_grad > 0
results['lora'] = {'trainable_parameters': sum(p.numel() for p in lora.parameters() if p.requires_grad),
                   'base_parameters': lora.base.weight.numel(), 'first_A_grad_norm': a_grad,
                   'first_B_grad_norm': b_grad}
print(results['lora'])


## 面试追问与实验记录
1. 手推 softmax+CE 的 p-y，解释 mean reduction。
2. 如果忘记 zero_grad，数值会怎样？
3. 为什么 eval 不阻断 gradient？
4. LoRA 的两个低秩因子为什么首步梯度不同？

记录 Question / Data / Config / Metric / Observation / Explanation / Limitation。
改变实验设置时不要反复用 test 选择答案；保存 notebook 后再下载 `.ipynb` 保留输出。


## 可选：导出指标
默认关闭。需要时打开开关；文件只写到当前 **Colab runtime** 的 `outputs/`。runtime 临时文件不等于已永久保存，可使用 Colab 文件面板下载。


In [ ]:
EXPORT_RESULTS = False
if EXPORT_RESULTS:
    destination = Path('outputs')
    destination.mkdir(exist_ok=True)
    (destination / 'metrics.json').write_text(
        json.dumps(results, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    print('Saved outputs/metrics.json in the Colab runtime; download it from the Files panel.')
